<a href="https://colab.research.google.com/github/andreupn/Context-integrity-assignment/blob/main/Contextual_Integrity_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [37]:
import pandas as pd

file_path = '/content/drive/MyDrive/CI_Dataset.csv'
try:
    df = pd.read_csv(file_path)
    print("Data loaded successfully:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path and try again.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

Data loaded successfully:


,State_Business_ID,Legal_Business_Name,Trade_Name,Entity_Type,Industry_Sector,NAICS_Code,EIN,Registration_Date,Business_Status,Street_Address,...,Owner_Full_Name,Owner_Title,Owner_Date_of_Birth,Owner_SSN_Masked,Owner_Phone,Owner_Email,Number_of_Employees,Annual_Revenue_USD,Zoning_Classification,Tax_Lien_Flag
0,PA-BUS-9406983,Summit Associates Corp.,Summit Associates,Corp.,Wholesale Trade,440673,42-4823830,1986-04-19,Active,4841 Washington Blvd,...,Jacob Edwards,Owner,1989-12-28,XXX-XX-9467,(570) 688-3791,jacob.edwards@summitassoci.com,5,236011,Agricultural,No
1,PA-BUS-7099530,Valley Management PC,Valley Management,PC,Retail Trade,807620,95-2086368,2006-04-10,Active,8615 Chestnut St,...,Gary James,Principal,1939-11-07,XXX-XX-6840,(724) 670-1199,gary.james@valleymanage.com,15,89775,Commercial,No
2,PA-BUS-4187908,Allegheny Supply LLP,Allegheny Supply,LLP,Insurance,177871,20-1939101,2013-10-17,Active,1110 Pine St,...,Richard Nelson,Principal,1955-04-23,XXX-XX-8472,(717) 241-5354,richard.nelson@alleghenysup.com,15,33944,Industrial,No
3,PA-BUS-8426297,Susquehanna Management Inc.,Susquehanna Management,Inc.,Insurance,784953,16-6349555,1989-04-16,Active,4006 Spring St,...,Linda Clark,President,1993-10-07,XXX-XX-8982,(267) 782-1157,linda.clark@susquehannam.com,100,35873,Industrial,No
4,PA-BUS-9180242,Heritage Logistics Corp.,Heritage Logistics,Corp.,Legal Services,117943,90-3961713,2006-09-16,Active,7485 Main St,...,Debra Watson,CEO,1967-05-22,XXX-XX-9242,(267) 840-8209,debra.watson@heritagelogi.com,25,79755,Mixed-Use,No


In [38]:
# Remove Direct Identifiers

direct_identifiers_to_remove = [
    'Owner_Full_Name',
    'Owner_SSN_Masked',
    'Legal_Business_Name',
    'Trade_Name',
    'Business_Phone',
    'Owner_Phone',
    'Business_Email',
    'Owner_Email',
    'EIN',
    'Street_Address'
]

columns_to_drop = [col for col in direct_identifiers_to_remove if col in df.columns]
if columns_to_drop:
    df = df.drop(columns=columns_to_drop)
    print(f"Removed direct identifier columns: {columns_to_drop}")
else:
    print("No direct identifier columns found to remove.")

display(df.head())

Removed direct identifier columns: ['Owner_Full_Name', 'Owner_SSN_Masked', 'Legal_Business_Name', 'Trade_Name', 'Business_Phone', 'Owner_Phone', 'Business_Email', 'Owner_Email', 'EIN', 'Street_Address']


,State_Business_ID,Entity_Type,Industry_Sector,NAICS_Code,Registration_Date,Business_Status,City,County,ZIP_Code,State,Owner_Title,Owner_Date_of_Birth,Number_of_Employees,Annual_Revenue_USD,Zoning_Classification,Tax_Lien_Flag
0,PA-BUS-9406983,Corp.,Wholesale Trade,440673,1986-04-19,Active,Harrisburg,Dauphin,17101,PA,Owner,1989-12-28,5,236011,Agricultural,No
1,PA-BUS-7099530,PC,Retail Trade,807620,2006-04-10,Active,Pittsburgh,Allegheny,15206,PA,Principal,1939-11-07,15,89775,Commercial,No
2,PA-BUS-4187908,LLP,Insurance,177871,2013-10-17,Active,Reading,Berks,19601,PA,Principal,1955-04-23,15,33944,Industrial,No
3,PA-BUS-8426297,Inc.,Insurance,784953,1989-04-16,Active,King of Prussia,Montgomery,19406,PA,President,1993-10-07,100,35873,Industrial,No
4,PA-BUS-9180242,Corp.,Legal Services,117943,2006-09-16,Active,Scranton,Lackawanna,18503,PA,CEO,1967-05-22,25,79755,Mixed-Use,No


In [40]:
# Transform Quasi-Identifiers

# Generalize ZIP_Code (e.g., 16802 -> 16***)
if 'ZIP_Code' in df.columns:
    df['ZIP_Code'] = df['ZIP_Code'].astype(str).str[:2] + '***'
    print("Generalized 'ZIP_Code'.")
else:
    print("'ZIP_Code' column not found for generalization.")

display(df.head())

Generalized 'ZIP_Code'.


,State_Business_ID,Entity_Type,Industry_Sector,NAICS_Code,Registration_Date,Business_Status,City,County,ZIP_Code,State,Owner_Title,Owner_Date_of_Birth,Number_of_Employees,Annual_Revenue_USD,Zoning_Classification,Tax_Lien_Flag
0,PA-BUS-9406983,Corp.,Wholesale Trade,440673,1986-04-19,Active,Harrisburg,Dauphin,17***,PA,Owner,1989-12-28,5,236011,Agricultural,No
1,PA-BUS-7099530,PC,Retail Trade,807620,2006-04-10,Active,Pittsburgh,Allegheny,15***,PA,Principal,1939-11-07,15,89775,Commercial,No
2,PA-BUS-4187908,LLP,Insurance,177871,2013-10-17,Active,Reading,Berks,19***,PA,Principal,1955-04-23,15,33944,Industrial,No
3,PA-BUS-8426297,Inc.,Insurance,784953,1989-04-16,Active,King of Prussia,Montgomery,19***,PA,President,1993-10-07,100,35873,Industrial,No
4,PA-BUS-9180242,Corp.,Legal Services,117943,2006-09-16,Active,Scranton,Lackawanna,18***,PA,CEO,1967-05-22,25,79755,Mixed-Use,No


In [41]:
# Bin Number_of_Employees

if 'Number_of_Employees' in df.columns:
    df['Number_of_Employees_Binned'] = df['Number_of_Employees'].apply(lambda x: 'Small (<250)' if x < 250 else 'Large (>=250)')
    print("Binned 'Number_of_Employees' into 'Small (<250)' and 'Large (>=250)'.")
    display(df[['Number_of_Employees', 'Number_of_Employees_Binned']].head())
else:
    print("'Number_of_Employees' column not found for binning.")

Binned 'Number_of_Employees' into 'Small (<250)' and 'Large (>=250)'.


,Number_of_Employees,Number_of_Employees_Binned
0,5,Small (<250)
1,15,Small (<250)
2,15,Small (<250)
3,100,Small (<250)
4,25,Small (<250)


In [42]:
# Bin Owner DOB

from datetime import datetime

if 'Owner_Date_of_Birth' in df.columns:
    # Convert 'Owner_Date_of_Birth' to datetime objects, coercing errors to NaT
    df['Owner_Date_of_Birth'] = pd.to_datetime(df['Owner_Date_of_Birth'], errors='coerce')

    # Calculate age based on a reference date (e.g., today's date)
    # For this example, let's use a fixed reference date for consistency, or datetime.now().year
    # Using a fixed date to ensure reproducibility if data is static
    reference_year = datetime.now().year

    # Calculate age. Handle NaT values introduced by 'coerce' during binning later
    df['Owner_Age'] = reference_year - df['Owner_Date_of_Birth'].dt.year

    # Define age bins and labels
    age_bins = [0, 35, 55, df['Owner_Age'].max() + 1]
    age_labels = ['Young (<35)', 'Middle-Aged (35-54)', 'Senior (55+)']

    # Bin the ages
    df['Owner_Age_Binned'] = pd.cut(df['Owner_Age'], bins=age_bins, labels=age_labels, right=False)

    print("Binned 'Owner_Date_of_Birth' into 'Owner_Age_Binned' categories.")
    display(df[['Owner_Date_of_Birth', 'Owner_Age', 'Owner_Age_Binned']].head())
else:
    print("'Owner_Date_of_Birth' column not found for binning.")

Binned 'Owner_Date_of_Birth' into 'Owner_Age_Binned' categories.


,Owner_Date_of_Birth,Owner_Age,Owner_Age_Binned
0,1989-12-28,37,Middle-Aged (35-54)
1,1939-11-07,87,Senior (55+)
2,1955-04-23,71,Senior (55+)
3,1993-10-07,33,Young (<35)
4,1967-05-22,59,Senior (55+)


In [43]:
# Bin Company age

from datetime import datetime

if 'Registration_Date' in df.columns:
    # Convert 'Registration_Date' to datetime objects, coercing errors to NaT
    df['Registration_Date'] = pd.to_datetime(df['Registration_Date'], errors='coerce')

    # Calculate company age based on a reference date (e.g., today's date)
    reference_date = datetime.now()

    # Calculate age in years
    df['Company_Age'] = (reference_date - df['Registration_Date']).dt.days / 365.25

    # Define age bins and labels
    company_age_bins = [0, 20, df['Company_Age'].max() + 1]
    company_age_labels = ['Early Stage (<20 years)', 'Mature Stage (20+ years)']

    # Bin the company ages
    df['Company_Age_Binned'] = pd.cut(df['Company_Age'], bins=company_age_bins, labels=company_age_labels, right=False)

    print("Binned 'Registration_Date' into 'Company_Age_Binned' categories.")
    display(df[['Registration_Date', 'Company_Age', 'Company_Age_Binned']].head())
else:
    print("'Registration_Date' column not found for binning.")

Binned 'Registration_Date' into 'Company_Age_Binned' categories.


,Registration_Date,Company_Age,Company_Age_Binned
0,1986-04-19,39.937029,Mature Stage (20+ years)
1,2006-04-10,19.961670,Early Stage (<20 years)
2,2013-10-17,12.440794,Early Stage (<20 years)
3,1989-04-16,36.944559,Mature Stage (20+ years)
4,2006-09-16,19.526352,Early Stage (<20 years)


In [44]:
# Bin Annual revenue

if 'Annual_Revenue_USD' in df.columns:
    # Define revenue bins and labels
    revenue_bins = [0, 500000, 5000000, df['Annual_Revenue_USD'].max() + 1]
    revenue_labels = ['<500K', '500K-5M', '>5M']

    # Bin the annual revenue
    df['Annual_Revenue_Binned'] = pd.cut(df['Annual_Revenue_USD'], bins=revenue_bins, labels=revenue_labels, right=False)

    print("Binned 'Annual_Revenue_USD' into 'Annual_Revenue_Binned' categories.")
    display(df[['Annual_Revenue_USD', 'Annual_Revenue_Binned']].head())
else:
    print("'Annual_Revenue_USD' column not found for binning.")

Binned 'Annual_Revenue_USD' into 'Annual_Revenue_Binned' categories.


,Annual_Revenue_USD,Annual_Revenue_Binned
0,236011,<500K
1,89775,<500K
2,33944,<500K
3,35873,<500K
4,79755,<500K


In [45]:
columns_to_remove_after_transformation = [
    'Number_of_Employees',
    'Owner_Date_of_Birth',
    'Owner_Age', # Intermediate column
    'Registration_Date',
    'Company_Age', # Intermediate column
    'Annual_Revenue_USD'
]

# Filter out columns that do not exist in the DataFrame to prevent errors
existing_columns_to_remove = [col for col in columns_to_remove_after_transformation if col in df.columns]

if existing_columns_to_remove:
    df = df.drop(columns=existing_columns_to_remove)
    print(f"Removed original transformed columns: {existing_columns_to_remove}")
else:
    print("No original transformed columns found to remove.")

print("DataFrame head after removing original columns:")
display(df.head())

Removed original transformed columns: ['Number_of_Employees', 'Owner_Date_of_Birth', 'Owner_Age', 'Registration_Date', 'Company_Age', 'Annual_Revenue_USD']
DataFrame head after removing original columns:


,State_Business_ID,Entity_Type,Industry_Sector,NAICS_Code,Business_Status,City,County,ZIP_Code,State,Owner_Title,Zoning_Classification,Tax_Lien_Flag,Number_of_Employees_Binned,Owner_Age_Binned,Company_Age_Binned,Annual_Revenue_Binned
0,PA-BUS-9406983,Corp.,Wholesale Trade,440673,Active,Harrisburg,Dauphin,17***,PA,Owner,Agricultural,No,Small (<250),Middle-Aged (35-54),Mature Stage (20+ years),<500K
1,PA-BUS-7099530,PC,Retail Trade,807620,Active,Pittsburgh,Allegheny,15***,PA,Principal,Commercial,No,Small (<250),Senior (55+),Early Stage (<20 years),<500K
2,PA-BUS-4187908,LLP,Insurance,177871,Active,Reading,Berks,19***,PA,Principal,Industrial,No,Small (<250),Senior (55+),Early Stage (<20 years),<500K
3,PA-BUS-8426297,Inc.,Insurance,784953,Active,King of Prussia,Montgomery,19***,PA,President,Industrial,No,Small (<250),Young (<35),Mature Stage (20+ years),<500K
4,PA-BUS-9180242,Corp.,Legal Services,117943,Active,Scranton,Lackawanna,18***,PA,CEO,Mixed-Use,No,Small (<250),Senior (55+),Early Stage (<20 years),<500K


In [47]:
#Removing 'ZIP_Code' for Further Generalization (as per previous decision)

if 'ZIP_Code' in df.columns:
    df = df.drop(columns=['ZIP_Code'])
    print("Successfully removed 'ZIP_Code' column.")
else:
    print("'ZIP_Code' column not found in DataFrame.")

display(df.head())

Successfully removed 'ZIP_Code' column.


,State_Business_ID,Entity_Type,Industry_Sector,NAICS_Code,Business_Status,City,County,State,Owner_Title,Zoning_Classification,Tax_Lien_Flag,Number_of_Employees_Binned,Owner_Age_Binned,Company_Age_Binned,Annual_Revenue_Binned
0,PA-BUS-9406983,Corp.,Wholesale Trade,440673,Active,Harrisburg,Dauphin,PA,Owner,Agricultural,No,Small (<250),Middle-Aged (35-54),Mature Stage (20+ years),<500K
1,PA-BUS-7099530,PC,Retail Trade,807620,Active,Pittsburgh,Allegheny,PA,Principal,Commercial,No,Small (<250),Senior (55+),Early Stage (<20 years),<500K
2,PA-BUS-4187908,LLP,Insurance,177871,Active,Reading,Berks,PA,Principal,Industrial,No,Small (<250),Senior (55+),Early Stage (<20 years),<500K
3,PA-BUS-8426297,Inc.,Insurance,784953,Active,King of Prussia,Montgomery,PA,President,Industrial,No,Small (<250),Young (<35),Mature Stage (20+ years),<500K
4,PA-BUS-9180242,Corp.,Legal Services,117943,Active,Scranton,Lackawanna,PA,CEO,Mixed-Use,No,Small (<250),Senior (55+),Early Stage (<20 years),<500K


In [58]:
#Calculate K-anonymity

quasi_identifiers_current = [
    'Number_of_Employees_Binned',
    'Owner_Age_Binned',
    'Company_Age_Binned',
    'Annual_Revenue_Binned'
]

# Filter for columns that actually exist in the DataFrame
existing_quasi_identifiers_current = [col for col in df.columns if col in quasi_identifiers_current]

if not existing_quasi_identifiers_current:
    print("No quasi-identifier columns found in df. Cannot compute k-anonymity.")
else:
    print(f"Using the following columns as quasi-identifiers for k-anonymity calculation: {existing_quasi_identifiers_current}")

    # Group by quasi-identifiers and count the size of each equivalence class
    equivalence_classes_current = df.groupby(existing_quasi_identifiers_current, observed=True).size().reset_index(name='count')

    # Calculate k-anonymity (minimum size of an equivalence class)
    k_anonymity_level_current = equivalence_classes_current['count'].min()

    print(f"\nThe k-anonymity level for the current DataFrame is: {k_anonymity_level_current}")

    print("\nPreview of equivalence classes (first 10):")
    display(equivalence_classes_current.head(10))

    if k_anonymity_level_current == 1:
        print("\nWarning: The k-anonymity level is 1, which means there is at least one unique record based on the quasi-identifiers. This indicates a high re-identification risk.")
    elif k_anonymity_level_current < 5:
        print(f"\nConsider increasing the k-anonymity level by further generalizing or transforming quasi-identifiers if {k_anonymity_level_current} is deemed too low for your privacy requirements.")

Using the following columns as quasi-identifiers for k-anonymity calculation: ['Number_of_Employees_Binned', 'Owner_Age_Binned', 'Company_Age_Binned', 'Annual_Revenue_Binned']

The k-anonymity level for the current DataFrame is: 1

Preview of equivalence classes (first 10):


,Number_of_Employees_Binned,Owner_Age_Binned,Company_Age_Binned,Annual_Revenue_Binned,count
0,Large (>=250),Young (<35),Mature Stage (20+ years),>5M,1
1,Large (>=250),Middle-Aged (35-54),Mature Stage (20+ years),<500K,1
2,Large (>=250),Senior (55+),Early Stage (<20 years),<500K,2
3,Large (>=250),Senior (55+),Early Stage (<20 years),500K-5M,2
4,Large (>=250),Senior (55+),Early Stage (<20 years),>5M,1
5,Large (>=250),Senior (55+),Mature Stage (20+ years),<500K,6
6,Large (>=250),Senior (55+),Mature Stage (20+ years),500K-5M,1
7,Small (<250),Young (<35),Early Stage (<20 years),<500K,14
8,Small (<250),Young (<35),Early Stage (<20 years),500K-5M,9
9,Small (<250),Young (<35),Early Stage (<20 years),>5M,1


In [59]:
unique_equivalence_classes = equivalence_classes_current[equivalence_classes_current['count'] < 6]

print("Unique records (count < 6) in equivalence classes:")
display(unique_equivalence_classes.head(20))

Unique records (count < 6) in equivalence classes:


,Number_of_Employees_Binned,Owner_Age_Binned,Company_Age_Binned,Annual_Revenue_Binned,count
0,Large (>=250),Young (<35),Mature Stage (20+ years),>5M,1
1,Large (>=250),Middle-Aged (35-54),Mature Stage (20+ years),<500K,1
2,Large (>=250),Senior (55+),Early Stage (<20 years),<500K,2
3,Large (>=250),Senior (55+),Early Stage (<20 years),500K-5M,2
4,Large (>=250),Senior (55+),Early Stage (<20 years),>5M,1
6,Large (>=250),Senior (55+),Mature Stage (20+ years),500K-5M,1
9,Small (<250),Young (<35),Early Stage (<20 years),>5M,1
15,Small (<250),Middle-Aged (35-54),Early Stage (<20 years),>5M,2


In [61]:
output_file_path = '/content/drive/MyDrive/anonymized_dataset.csv'
df.to_csv(output_file_path, index=False)
print(f"Anonymized dataset saved to: {output_file_path}")

Anonymized dataset saved to: /content/drive/MyDrive/anonymized_dataset.csv


In [51]:
print("\n--- Quantifying Information Loss for Number_of_Employees ---")

if 'Number_of_Employees' in df_original.columns:
    original_unique_emp = df_original['Number_of_Employees'].nunique()
    original_min_emp = df_original['Number_of_Employees'].min()
    original_max_emp = df_original['Number_of_Employees'].max()

    print(f"Original 'Number_of_Employees':\n  Unique values: {original_unique_emp}\n  Min value: {original_min_emp}\n  Max value: {original_max_emp}")
    print("\nOriginal 'Number_of_Employees' value counts:")
    display(df_original['Number_of_Employees'].value_counts().sort_index())

    if 'Number_of_Employees_Binned' in df.columns:
        print("\nAnonymized 'Number_of_Employees_Binned' value counts:")
        display(df['Number_of_Employees_Binned'].value_counts().sort_index())
    else:
        print("Anonymized 'Number_of_Employees_Binned' column not found in current df.")
else:
    print("Original 'Number_of_Employees' column not found in df_original.")



--- Quantifying Information Loss for Number_of_Employees ---
Original 'Number_of_Employees':
  Unique values: 18
  Min value: 1
  Max value: 500

Original 'Number_of_Employees' value counts:


,count
Number_of_Employees,
1,18
2,14
3,19
4,17
5,22
7,18
10,15
15,23
20,15



Anonymized 'Number_of_Employees_Binned' value counts:


,count
Number_of_Employees_Binned,
Large (>=250),14
Small (<250),286


In [53]:
print("\n--- Quantifying Information Loss for Owner_Date_of_Birth ---")

if 'Owner_Date_of_Birth' in df_original.columns:
    original_unique_dob = df_original['Owner_Date_of_Birth'].nunique()
    print(f"Original 'Owner_Date_of_Birth':\n  Unique values: {original_unique_dob}")

    # Convert to datetime and calculate age for original for comparison
    df_original_temp = df_original.copy()
    df_original_temp['Owner_Date_of_Birth'] = pd.to_datetime(df_original_temp['Owner_Date_of_Birth'], errors='coerce')
    reference_year = datetime.now().year
    df_original_temp['Owner_Age'] = reference_year - df_original_temp['Owner_Date_of_Birth'].dt.year

    original_min_age = df_original_temp['Owner_Age'].min()
    original_max_age = df_original_temp['Owner_Age'].max()
    print(f"  Min calculated age: {original_min_age}\n  Max calculated age: {original_max_age}")

    if 'Owner_Age_Binned' in df.columns:
        print("\nAnonymized 'Owner_Age_Binned' value counts:")
        display(df['Owner_Age_Binned'].value_counts().sort_index())
    else:
        print("Anonymized 'Owner_Age_Binned' column not found in current df.")
else:
    print("Original 'Owner_Date_of_Birth' column not found in df_original.")


--- Quantifying Information Loss for Owner_Date_of_Birth ---
Original 'Owner_Date_of_Birth':
  Unique values: 300
  Min calculated age: 20
  Max calculated age: 102

Anonymized 'Owner_Age_Binned' value counts:


,count
Owner_Age_Binned,
Young (<35),56
Middle-Aged (35-54),69
Senior (55+),175


In [54]:
print("\n--- Quantifying Information Loss for Registration_Date ---")

if 'Registration_Date' in df_original.columns:
    original_unique_reg_date = df_original['Registration_Date'].nunique()
    print(f"Original 'Registration_Date':\n  Unique values: {original_unique_reg_date}")

    # Convert to datetime and calculate company age for original for comparison
    df_original_temp = df_original.copy()
    df_original_temp['Registration_Date'] = pd.to_datetime(df_original_temp['Registration_Date'], errors='coerce')
    reference_date = datetime.now()
    df_original_temp['Company_Age'] = (reference_date - df_original_temp['Registration_Date']).dt.days / 365.25

    original_min_company_age = df_original_temp['Company_Age'].min()
    original_max_company_age = df_original_temp['Company_Age'].max()
    print(f"  Min calculated company age: {original_min_company_age:.2f} years\n  Max calculated company age: {original_max_company_age:.2f} years")

    if 'Company_Age_Binned' in df.columns:
        print("\nAnonymized 'Company_Age_Binned' value counts:")
        display(df['Company_Age_Binned'].value_counts().sort_index())
    else:
        print("Anonymized 'Company_Age_Binned' column not found in current df.")
else:
    print("Original 'Registration_Date' column not found in df_original.")


--- Quantifying Information Loss for Registration_Date ---
Original 'Registration_Date':
  Unique values: 300
  Min calculated company age: 1.25 years
  Max calculated company age: 41.10 years

Anonymized 'Company_Age_Binned' value counts:


,count
Company_Age_Binned,
Early Stage (<20 years),135
Mature Stage (20+ years),165


In [60]:
print("\n--- Quantifying Information Loss for Annual_Revenue_USD ---")

if 'Annual_Revenue_USD' in df_original.columns:
    original_unique_revenue = df_original['Annual_Revenue_USD'].nunique()
    original_min_revenue = df_original['Annual_Revenue_USD'].min()
    original_max_revenue = df_original['Annual_Revenue_USD'].max()

    print(f"Original 'Annual_Revenue_USD':\n  Unique values: {original_unique_revenue}\n  Min value: {original_min_revenue}\n  Max value: {original_max_revenue}")
    print("\nOriginal 'Annual_Revenue_USD' value counts (top 10):")
    display(df_original['Annual_Revenue_USD'].value_counts().sort_index().head(10))

    if 'Annual_Revenue_Binned' in df.columns:
        print("\nAnonymized 'Annual_Revenue_Binned' value counts:")
        display(df['Annual_Revenue_Binned'].value_counts().sort_index())
    else:
        print("Anonymized 'Annual_Revenue_Binned' column not found in current df.")
else:
    print("Original 'Annual_Revenue_USD' column not found in df_original.")


--- Quantifying Information Loss for Annual_Revenue_USD ---
Original 'Annual_Revenue_USD':
  Unique values: 300
  Min value: 10504
  Max value: 97022836

Original 'Annual_Revenue_USD' value counts (top 10):


,count
Annual_Revenue_USD,
10504,1
10827,1
13692,1
14187,1
14725,1
16845,1
17266,1
17994,1
20644,1



Anonymized 'Annual_Revenue_Binned' value counts:


,count
Annual_Revenue_Binned,
<500K,189
500K-5M,80
>5M,31
